In [1]:
from pathlib import Path
import numpy as np
from glob import glob
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
import pandas as pd
from itertools import product
from evaluation.test_datasets import FaceRecogntionDataset

In [2]:
# ds = FaceRecogntionDataset(dataset_name='small_120-perspk1', dataset_path='/app/datasets/vb-clean_ident/Small_120/perspk1')

In [3]:
# predictions = np.load("/app/datasets/vb-clean_ident/scf_vb2.npz")
# kappa = predictions["unc"][:, 0]
# sns.histplot(data=kappa, kde=True)

## Create meta files

In [4]:
# there is not templates for out-of-gallery samples. They all come as single template image

In [5]:
ident_ds_dir = Path("/app/datasets/vb-clean_ident")
ident_ds_dir.mkdir(exist_ok=True)

# list protocols
protools_path = "/app/sandbox/ScriptsForVoxBlink2/data/ossi"
protools_paths = list(glob(protools_path + "/*"))

# create meta files
template_idx_shift = 10000  # shift to get unique template ids
for protocol_path, template_size in product(
    protools_paths, ["perspk1", "perspk3", "perspk5"]
):
    protocol_path = Path(protocol_path)
    ds_name = (protocol_path.parts[-1] + "-" + template_size).lower()
    meta_path = ident_ds_dir / protocol_path.parts[-1] / template_size / "meta"
    meta_path.mkdir(exist_ok=True, parents=True)

    ossi_info_path = protocol_path / template_size
    gallery_ids = []
    gallery_paths = []
    with open(ossi_info_path / "gallery") as fd:
        for line in fd:
            spk_id = int(line.split(" ")[-1][:-1])
            gallery_ids.append(spk_id)
            audio_path_parts = line.split(" ")[0]
            audio_path = (
                audio_path_parts[:7]
                + "/"
                + audio_path_parts[8:19]
                + "/"
                + audio_path_parts[-5:]
                + ".wav"
            )
            gallery_paths.append(audio_path)
    gallery_ids = np.array(gallery_ids)

    probe_ids = []
    probe_paths = []
    with open(ossi_info_path / "probe") as fd:
        for line in fd:
            spk_id = int(line.split(" ")[-1][:-1])
            probe_ids.append(spk_id)
            audio_path_parts = line.split(" ")[0]
            audio_path = (
                audio_path_parts[:7]
                + "/"
                + audio_path_parts[8:19]
                + "/"
                + audio_path_parts[-5:]
                + ".wav"
            )
            probe_paths.append(audio_path)
    probe_ids = np.array(probe_ids)

    # split out-of-gallery probes
    # the out-of-gallery set can also be split into templates using class ids
    oog_id = probe_ids[-1]
    oog_index_start = np.nonzero(probe_ids == oog_id)[0][0]
    # probe_template_ids = np.copy(probe_ids)
    # probe_template_ids[oog_index_start:] += np.arange(len(probe_ids[oog_index_start:]))
    # probe_template_ids += template_idx_shift
    probe_proper_ids = np.copy(probe_ids)[:oog_index_start]
    true_ids = [probe_path.split("/")[0] for probe_path in probe_paths][
        oog_index_start:
    ]
    id_change_counter = -1
    prev_id = None
    probe_proper_ids_oog = []
    for true_id in true_ids:
        if prev_id != true_id:
            prev_id = true_id
            id_change_counter += 1
        probe_proper_ids_oog.append(oog_id + id_change_counter)
    probe_proper_ids = np.concatenate(
        [probe_proper_ids, np.array(probe_proper_ids_oog)]
    )
    probe_template_ids = probe_proper_ids + template_idx_shift

    # create tid/mid file
    audio_paths = gallery_paths + probe_paths
    ids = np.concatenate([gallery_ids, probe_proper_ids])
    tids = np.concatenate([gallery_ids, probe_template_ids])
    mids = np.arange(len(ids))
    out_file_tid_mid = meta_path / Path(f"{ds_name}_face_tid_mid.txt")
    with open(out_file_tid_mid, "w") as fd:
        for name, tid, sid, mid in zip(audio_paths, tids, ids, mids):
            fd.write(f"{name} {tid} {mid} {sid}\n")

    # create gallery and probe meta files
    out_file_probe = meta_path / Path(f"{ds_name}_1N_probe_mixed.csv")
    out_file_gallery = meta_path / Path(f"{ds_name}_1N_gallery_G1.csv")

    assert len(gallery_ids) + len(probe_template_ids) == len(audio_paths)
    probe = pd.DataFrame(
        {
            "TEMPLATE_ID": probe_template_ids,
            "SUBJECT_ID": probe_proper_ids,
            "FILENAME": probe_paths,
        }
    )
    gallery = pd.DataFrame(
        {
            "TEMPLATE_ID": gallery_ids,
            "SUBJECT_ID": gallery_ids,
            "FILENAME": gallery_paths,
        }
    )

    probe.to_csv(out_file_probe, sep=",", index=False)
    gallery.to_csv(out_file_gallery, sep=",", index=False)

## Put embeddings in corresponding dirs

In [6]:
for embs_name in ["scf", "pfe", "scaleface"]:
    # embs_name = 'scf' #"pfe_vb2"  # embs_name = "scaleface_vb2" # #embs_name = "scf_vb2"
    all_embeddings = np.load(f"/app/datasets/vb-clean_ident/{embs_name}.npz")
    unc = all_embeddings["unc"]
    embs = all_embeddings["embs"]
    all_paths = np.load("/app/datasets/vb-clean_ident/paths.npy")
    path_to_id = {"/".join(path.split("/")[-3:]): i for i, path in enumerate(all_paths)}

    ident_ds_dir = Path("/app/datasets/vb-clean_ident")
    ident_ds_dir.mkdir(exist_ok=True)

    protools_path = "/app/sandbox/ScriptsForVoxBlink2/data/ossi"
    protools_paths = list(glob(protools_path + "/*"))
    for protocol_path, template_size in tqdm(
        product(protools_paths, ["perspk1", "perspk3", "perspk5"])
    ):
        protocol_path = Path(protocol_path)
        ds_name = (protocol_path.parts[-1] + "-" + template_size).lower()
        meta_path = ident_ds_dir / protocol_path.parts[-1] / template_size / "meta"
        embeddings_dir_path = (
            ident_ds_dir / protocol_path.parts[-1] / template_size / "embeddings"
        )
        embeddings_dir_path.mkdir(exist_ok=True, parents=True)
        tid_mid_path = meta_path / Path(f"{ds_name}_face_tid_mid.txt")
        file_names = []
        with open(tid_mid_path) as fd:
            for line in fd:
                file_names.append(line.split(" ")[0])

        embs_part = []
        unc_part = []
        for file_name in file_names:
            idx = path_to_id[file_name]
            embs_part.append(embs[idx][np.newaxis, :])
            unc_part.append(unc[idx][np.newaxis, :])
        embs_part = np.concatenate(embs_part, axis=0)
        unc_part = np.concatenate(unc_part, axis=0)
        np.savez(
            embeddings_dir_path / f"{embs_name}_embs_{ds_name}.npz",
            embs=embs_part,
            unc=unc_part,
        )

9it [00:09,  1.01s/it]
9it [00:11,  1.32s/it]
9it [00:09,  1.00s/it]
